Los Cuatro Pilares de la Programación Orientada a Objetos (POO)

https://gist.github.com/robintux/a4d06f6e2bba46537ea78070c0dd662e

In [1]:
from __future__ import annotations
from typing import List, Optional, Union
from math import nan
import json
from statistics import mean, median, stdev

# --- Jerarquía de Calificaciones ---

class Calificacion:
    """Representa una calificación con metadatos (extensible)."""
    def __init__(self, valor: float, asignatura: Optional[str] = None, fecha: Optional[str] = None):
        if not (0 <= valor <= 10):
            raise ValueError("La calificación debe estar entre 0 y 10.")
        if not isinstance(valor, (int, float)):
            raise TypeError("La calificación debe ser numérica.")
        self.valor = float(valor)
        self.asignatura = asignatura
        self.fecha = fecha

    def contribucion_al_promedio(self) -> float:
        """Polimórfico: contribución al cálculo de promedio."""
        return self.valor

    def ponderacion(self) -> float:
        """Polimórfico: ponderación usada en promedio ponderado."""
        return 1.0  # Por defecto, cuenta como 1 unidad

    def to_dict(self):
        return {"tipo": self.__class__.__name__, "valor": self.valor, "asignatura": self.asignatura, "fecha": self.fecha}

    def __repr__(self):
        return f"Calificacion(valor={self.valor}, asignatura={self.asignatura})"

class CalificacionPonderada(Calificacion):
    """Calificación con ponderación (ej: parcial 1 vale 30% del total)."""
    def __init__(self, valor: float, peso: float, asignatura: Optional[str] = None, fecha: Optional[str] = None):
        super().__init__(valor, asignatura, fecha)
        if not (0 <= peso <= 1):
            raise ValueError("El peso debe estar entre 0 y 1.")
        self.peso = peso

    def contribucion_al_promedio(self) -> float:
        # Polimórfico: la contribución es valor * peso
        return self.valor * self.peso

    def ponderacion(self) -> float:
        # Polimórfico: la ponderación es el peso
        return self.peso

    def to_dict(self):
        data = super().to_dict()
        data["peso"] = self.peso
        return data

    def __repr__(self):
        return f"CalificacionPonderada(valor={self.valor}, peso={self.peso}, asignatura={self.asignatura})"

class CalificacionFinal(CalificacionPonderada):
    """Calificación de un examen final o trabajo final, con aprobación automática."""
    def __init__(self, valor: float, peso: float, asignatura: str, fecha: Optional[str] = None, aprobatorio: bool = True):
        super().__init__(valor, peso, asignatura, fecha)
        self.aprobatorio = aprobatorio

    def es_aprobado(self) -> bool:
        return self.valor >= 6.0 and self.aprobatorio

    def to_dict(self):
        data = super().to_dict()
        data["aprobatorio"] = self.aprobatorio
        data["aprobado"] = self.es_aprobado()
        return data

    def __repr__(self):
        return f"CalificacionFinal(valor={self.valor}, peso={self.peso}, aprobado={self.es_aprobado()})"

# --- Jerarquía de Estudiantes ---

class Estudiante:
    """Representa a un estudiante con nombre, edad y calificaciones estructuradas."""
    def __init__(self, nombre: str, edad: int):
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValueError("El nombre debe ser una cadena no vacía.")
        if not isinstance(edad, int) or edad < 0:
            raise ValueError("La edad debe ser un entero no negativo.")

        self.nombre = nombre.strip()
        self.edad = edad
        self._calificaciones: List[Calificacion] = []

    def agregar_calificacion(self, calif: Calificacion) -> None:
        # Recibe un objeto Calificacion (puede ser Simple, Ponderada, Final, etc.)
        self._calificaciones.append(calif)

    def promedio(self) -> float:
        """Polimórfico: calcula el promedio según la lógica del tipo de estudiante."""
        if not self._calificaciones:
            return nan
        return sum(c.contribucion_al_promedio() for c in self._calificaciones) / len(self._calificaciones)

    def estadisticas_descriptivas(self) -> dict:
        """Polimórfico: estadísticas que pueden adaptarse según el tipo de estudiante."""
        valores = [c.valor for c in self._calificaciones]
        if not valores:
            return {"promedio": nan, "mediana": nan, "desviacion_estandar": nan}
        return {
            "promedio": mean(valores),
            "mediana": median(valores),
            "desviacion_estandar": stdev(valores) if len(valores) > 1 else 0.0
        }

    def to_dict(self) -> dict:
        """Polimórfico: serialización que puede adaptarse por tipo de estudiante."""
        return {
            "tipo": self.__class__.__name__,
            "nombre": self.nombre,
            "edad": self.edad,
            "calificaciones": [c.to_dict() for c in self._calificaciones],
            "promedio": self.promedio()  # Llama al método polimórfico
        }

    def __repr__(self):
        return f"Estudiante(nombre='{self.nombre}', edad={self.edad}, n_calif={len(self._calificaciones)})"

class EstudianteUniversitario(Estudiante):
    """Estudiante universitario que puede tener calificaciones ponderadas."""
    def promedio(self) -> float:
        """Polimórfico: implementación de promedio ponderado."""
        if not self._calificaciones:
            return nan
        total_valor = sum(c.contribucion_al_promedio() for c in self._calificaciones)
        total_peso = sum(c.ponderacion() for c in self._calificaciones)
        return total_valor / total_peso if total_peso > 0 else nan

    def __repr__(self):
        return f"EstudianteUniversitario(nombre='{self.nombre}', edad={self.edad}, n_calif={len(self._calificaciones)})"

class EstudianteBecado(EstudianteUniversitario):
    """Estudiante con beca que requiere un promedio mínimo."""
    def __init__(self, nombre: str, edad: int, promedio_minimo: float = 8.0):
        super().__init__(nombre, edad)
        self.promedio_minimo = promedio_minimo

    @property
    def beca_activa(self) -> bool:
        return self.promedio() >= self.promedio_minimo  # Llama al método polimórfico de su padre

    def to_dict(self) -> dict:
        """Polimórfico: extiende serialización para incluir estado de beca."""
        data = super().to_dict()
        data["beca_activa"] = self.beca_activa
        return data

    def __repr__(self):
        estado = "ACTIVA" if self.beca_activa else "SUSPENDIDA"
        return f"EstudianteBecado(nombre='{self.nombre}', edad={self.edad}, beca={estado})"

class EstudianteSecundaria(Estudiante):
    """Estudiante de secundaria con lógica de promoción simple."""
    def promociona(self) -> bool:
        return self.promedio() >= 6.0 and all(c.valor >= 4.0 for c in self._calificaciones)

    def to_dict(self) -> dict:
        """Polimórfico: extiende serialización para incluir estado de promoción."""
        data = super().to_dict()
        data["promociona"] = self.promociona()
        return data

    def __repr__(self):
        estado = "Promociona" if self.promociona() else "No promociona"
        return f"EstudianteSecundaria(nombre='{self.nombre}', edad={self.edad}, estado={estado})"

In [2]:
# Ejemplos de uso

# Creamos estudiantes de diferentes tipos
estudiantes = [
    EstudianteSecundaria("Ana López", 15),
    EstudianteUniversitario("Carlos Méndez", 20),
    EstudianteBecado("Diana Ruiz", 21)
]

In [3]:

# Agregamos calificaciones de diferentes tipos
estudiantes[0].agregar_calificacion(Calificacion(8.0, "Matemáticas"))
estudiantes[0].agregar_calificacion(Calificacion(7.5, "Lengua"))

estudiantes[1].agregar_calificacion(CalificacionPonderada(9.0, peso=0.3, asignatura="Álgebra"))
estudiantes[1].agregar_calificacion(CalificacionPonderada(7.5, peso=0.7, asignatura="Cálculo"))

estudiantes[2].agregar_calificacion(CalificacionPonderada(9.2, peso=0.4, asignatura="Física"))
estudiantes[2].agregar_calificacion(CalificacionPonderada(8.8, peso=0.6, asignatura="Química"))

In [4]:
# Iteramos y vemos cómo se comporta `promedio()` de forma polimórfica
for est in estudiantes:
    print(est)
    print(f"  Promedio calculado: {est.promedio():.2f}")  # <-- Polimorfismo en acción
    print(f"  Serialización: {est.to_dict()}")           # <-- Polimorfismo en serialización
    print("-" * 50)

EstudianteSecundaria(nombre='Ana López', edad=15, estado=Promociona)
  Promedio calculado: 7.75
  Serialización: {'tipo': 'EstudianteSecundaria', 'nombre': 'Ana López', 'edad': 15, 'calificaciones': [{'tipo': 'Calificacion', 'valor': 8.0, 'asignatura': 'Matemáticas', 'fecha': None}, {'tipo': 'Calificacion', 'valor': 7.5, 'asignatura': 'Lengua', 'fecha': None}], 'promedio': 7.75, 'promociona': True}
--------------------------------------------------
EstudianteUniversitario(nombre='Carlos Méndez', edad=20, n_calif=2)
  Promedio calculado: 7.95
  Serialización: {'tipo': 'EstudianteUniversitario', 'nombre': 'Carlos Méndez', 'edad': 20, 'calificaciones': [{'tipo': 'CalificacionPonderada', 'valor': 9.0, 'asignatura': 'Álgebra', 'fecha': None, 'peso': 0.3}, {'tipo': 'CalificacionPonderada', 'valor': 7.5, 'asignatura': 'Cálculo', 'fecha': None, 'peso': 0.7}], 'promedio': 7.949999999999999}
--------------------------------------------------
EstudianteBecado(nombre='Diana Ruiz', edad=21, beca=AC